# Libraries

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(_root / "src"))
from tfm.utils import ROOT_PATH, DATA_PATH, PROCESSED_PATH
from tfm.geo import normalize_mun_code

np.random.seed(42)

# Data Loading

In [ ]:
# Load users and processed tables from 02_geographic_enrichment
df_users         = pd.read_csv(PROCESSED_PATH / "users_base.csv", encoding="utf-8", dtype={"cp_num": str}, low_memory=False)
df_municipios    = pd.read_csv(PROCESSED_PATH / "mun_enriched.csv",  low_memory=False)
df_prov_enriched = pd.read_csv(PROCESSED_PATH / "prov_enriched.csv")
df_cp_mun        = pd.read_csv(PROCESSED_PATH / "cp_municipio.csv", dtype={"cp_code": str})
df_sector        = pd.read_csv(PROCESSED_PATH / "sector_mun.csv")
df_bcn_household = pd.read_csv(PROCESSED_PATH / "bcn_household.csv")
df_mad_household = pd.read_csv(PROCESSED_PATH / "mad_household.csv")
df_mun_household = pd.read_csv(PROCESSED_PATH / "mun_household.csv")

# Ensure correct types in key columns
df_municipios["mun_code"]      = pd.to_numeric(df_municipios["mun_code"],      errors="coerce").astype("Int64")
df_municipios["mun_code_base"] = pd.to_numeric(df_municipios["mun_code_base"], errors="coerce").astype("Int64")
df_cp_mun["mun_code"]          = pd.to_numeric(df_cp_mun["mun_code"],          errors="coerce").astype("Int64")
df_sector["mun_code_base"]     = pd.to_numeric(df_sector["mun_code_base"],     errors="coerce").astype("Int64")

print(f"usuarios:      {df_users.shape}")
print(f"municipios:    {df_municipios.shape}")
print(f"prov_enriched: {df_prov_enriched.shape}")
print(f"cp_municipio:  {df_cp_mun.shape}")
print(f"sector_mun:    {df_sector.shape}")

---
# User Enrichment

## Territorial Context

In [ ]:
df_mun_rates   = df_municipios.merge(df_prov_enriched, on="prov_code", how="left")
df_cp_mun_full = df_cp_mun.merge(df_mun_rates, on="mun_code", how="left")

print(f"df_mun_rates:   {df_mun_rates.shape}")
print(f"df_cp_mun_full: {df_cp_mun_full.shape}")

check = df_municipios[df_municipios["mun_name"] == "Barcelona"][["mun_code", "district_name", "prov_code"]]
display(check.head(4))

In [ ]:
df_users["cp_code"] = df_users["cp_num"].astype("string")

COLS_JOIN = [
    "cp_code", "mun_code", "mun_code_base", "prov_code",
    "mun_name", "district_name", "prob_car",
    "ipa_value", "total_people", "distance_km",
    "rate_activity", "rate_paro",
    "rate_student_inactive", "rate_disabled_inactive",
    "rate_retired_inactive", "rate_household_inactive", "rate_other_inactive",
    "rate_soltero", "rate_casado", "rate_viudo", "rate_divorciado",
    "rate_hab_menos3", "rate_hab_3a6", "rate_hab_7mas",
]

df_users_geo = df_users.merge(
    df_cp_mun_full[COLS_JOIN],
    on="cp_code",
    how="inner"
)

df_users_geo["mun_code_base"] = df_users_geo["mun_code_base"].astype("Int64")

# Provincial fallback: for users without a CP->municipality match,
# derive prov_code from the first 2 digits of the CP (Spanish convention)
# and fill in provincial rates + the provincial mean prob_car
mask_sin_prov = df_users_geo["prov_code"].isna()

df_users_geo.loc[mask_sin_prov, "prov_code"] = (
    pd.to_numeric(df_users_geo.loc[mask_sin_prov, "cp_num"].str[:2], errors="coerce")
)
df_users_geo["prov_code"] = pd.to_numeric(df_users_geo["prov_code"], errors="coerce").astype("Int64")

# Fill in provincial rates for users without a municipality
RATE_COLS = [
    "rate_activity", "rate_paro",
    "rate_student_inactive", "rate_disabled_inactive",
    "rate_retired_inactive", "rate_household_inactive", "rate_other_inactive",
    "rate_soltero", "rate_casado", "rate_viudo", "rate_divorciado",
    "rate_hab_menos3", "rate_hab_3a6", "rate_hab_7mas",
]

prov_rates_idx = df_prov_enriched.set_index("prov_code")
for col in RATE_COLS:
    null_mask = df_users_geo[col].isna() & df_users_geo["prov_code"].notna()
    if null_mask.any():
        df_users_geo.loc[null_mask, col] = (
            df_users_geo.loc[null_mask, "prov_code"].map(prov_rates_idx[col])
        )

# prob_car: provincial mean as fallback
prob_car_prov = df_municipios.groupby("prov_code")["prob_car"].mean()
null_prob = df_users_geo["prob_car"].isna() & df_users_geo["prov_code"].notna()
df_users_geo.loc[null_prob, "prob_car"] = (
    df_users_geo.loc[null_prob, "prov_code"].map(prob_car_prov)
)

n_sin_geo = mask_sin_prov.sum()
print(f"Usuarios con equivalencia CP→municipio: {(~mask_sin_prov).sum():,} de {len(df_users):,}")
print(f"Usuarios con fallback provincial:        {n_sin_geo:,}")
display(df_users_geo[["id_user", "cp_num", "mun_name", "prov_code", "ipa_value", "distance_km"]].head(5))

In [ ]:
# Impute gender for users without a value: Bernoulli(man/total_people) per municipality
df_gender_prob = (
    df_municipios[["mun_code", "man", "total_people"]]
    .assign(prob_male=lambda d: d["man"] / d["total_people"])
    [["mun_code", "prob_male"]]
)

df_users_geo = df_users_geo.merge(df_gender_prob, on="mun_code", how="left")

# 1) Normalise the OBSERVED gender to H (male) / M (female).
#    Dataset convention: M=Male->H, F=Female->M, H=Hombre->H.
#    Empty or unrecognised values remain NaN and are imputed below.
df_users_geo["gender"] = df_users_geo["gender"].map({"H": "H", "M": "H", "F": "M"})

# 2) Impute the missing ones with Bernoulli(prob_male) per municipality, generating
#    the final H/M labels DIRECTLY. They are not remapped again, so that
#    imputed women remain as "M" (previously a subsequent .map() wrongly
#    converted them into "H").
missing_gender = df_users_geo["gender"].isna()
has_prob = df_users_geo["prob_male"].notna()

idx = missing_gender & has_prob
df_users_geo.loc[idx, "gender"] = np.where(
    np.random.random(idx.sum()) < df_users_geo.loc[idx, "prob_male"].to_numpy(),
    "H", "M"
)

df_users_geo = df_users_geo.drop(columns=["prob_male"])

print("Género tras normalización:")
print(df_users_geo["gender"].value_counts())
print(f"Sin género: {df_users_geo['gender'].isna().sum()}")

## Labour Model

In [ ]:
n = len(df_users_geo)

has_activity_rate = df_users_geo["rate_activity"].notna()
has_paro_rate     = df_users_geo["rate_paro"].notna()

is_active     = np.zeros(n, dtype=bool)
is_unemployed = np.zeros(n, dtype=bool)

idx_activity = np.where(has_activity_rate)[0]
is_active[idx_activity] = (
    np.random.random(len(idx_activity))
    < df_users_geo.loc[idx_activity, "rate_activity"].to_numpy()
)

idx_paro = np.where(is_active & has_paro_rate)[0]
is_unemployed[idx_paro] = (
    np.random.random(len(idx_paro))
    < df_users_geo.loc[idx_paro, "rate_paro"].to_numpy()
)

df_users_geo["labor_status"] = "unknown"
df_users_geo.loc[has_activity_rate & ~is_active,  "labor_status"] = "inactive"
df_users_geo.loc[is_active & is_unemployed,        "labor_status"] = "unemployed"
df_users_geo.loc[is_active & ~is_unemployed,       "labor_status"] = "employed"

print(df_users_geo["labor_status"].value_counts())

In [ ]:

df_sector_agg = (
    df_sector
    .dropna(subset=["mun_code_base"])
    .groupby(["mun_code_base", "sector"], as_index=False)["valor"]
    .sum()
)
df_sector_agg["prob_sector"] = (
    df_sector_agg["valor"]
    / df_sector_agg.groupby("mun_code_base")["valor"].transform("sum")
).fillna(0)


def sample_sector_by_mun(group):
    sec = df_sector_agg.loc[df_sector_agg["mun_code_base"] == group.name]
    if sec.empty:
        return pd.Series([pd.NA] * len(group), index=group.index)
    probs = sec["prob_sector"].to_numpy()
    total = probs.sum()
    if total == 0:
        return pd.Series([pd.NA] * len(group), index=group.index)
    return pd.Series(
        np.random.choice(sec["sector"].to_numpy(), size=len(group), p=probs / total),
        index=group.index
    )


df_users_geo["labor_sector"] = pd.NA
employed_mask = df_users_geo["labor_status"] == "employed"

df_users_geo.loc[employed_mask, "labor_sector"] = (
    df_users_geo.loc[employed_mask]
    .groupby("mun_code_base", group_keys=False)
    .apply(sample_sector_by_mun)
)

print(df_users_geo.loc[employed_mask, "labor_sector"].value_counts())

In [ ]:
INACTIVE_COLS   = [
    "rate_student_inactive", "rate_disabled_inactive",
    "rate_retired_inactive",  "rate_household_inactive", "rate_other_inactive",
]
INACTIVE_LABELS = ["student", "disabled", "retired", "household", "other"]


def sample_inactive_type_by_prov(group):
    prov  = group["prov_code"].iloc[0]
    rates = df_prov_enriched.loc[df_prov_enriched["prov_code"] == prov, INACTIVE_COLS]
    if rates.empty:
        return pd.Series([pd.NA] * len(group), index=group.index)
    probs = rates.iloc[0].fillna(0).to_numpy()
    total = probs.sum()
    if total == 0:
        return pd.Series([pd.NA] * len(group), index=group.index)
    return pd.Series(
        np.random.choice(INACTIVE_LABELS, size=len(group), p=probs / total),
        index=group.index
    )


df_users_geo["inactive_type"] = pd.NA
inactive_mask = df_users_geo["labor_status"] == "inactive"

df_users_geo.loc[inactive_mask, "inactive_type"] = (
    df_users_geo.loc[inactive_mask]
    .groupby("mun_code_base", group_keys=False)
    .apply(sample_inactive_type_by_prov)
)

df_users_geo["labor_type"] = pd.NA
df_users_geo.loc[df_users_geo["labor_status"] == "employed",   "labor_type"] = df_users_geo["labor_sector"]
df_users_geo.loc[df_users_geo["labor_status"] == "inactive",   "labor_type"] = df_users_geo["inactive_type"]
df_users_geo.loc[df_users_geo["labor_status"] == "unemployed", "labor_type"] = "unemployed"
df_users_geo.loc[df_users_geo["labor_type"].isna(),            "labor_type"] = "unknown_no_context"

df_users_geo = df_users_geo.drop(columns=["labor_sector", "inactive_type"])

print(df_users_geo["labor_type"].value_counts())

## Marital Status

In [ ]:
CIVIL_COLS   = ["rate_soltero", "rate_casado", "rate_viudo", "rate_divorciado"]
CIVIL_LABELS = ["soltero",      "casado",      "viudo",      "divorciado"]


def sample_civil_by_prov(group):
    prov  = group["prov_code"].iloc[0]
    rates = df_prov_enriched.loc[df_prov_enriched["prov_code"] == prov, CIVIL_COLS]
    if rates.empty:
        return pd.Series([pd.NA] * len(group), index=group.index)
    probs = rates.iloc[0].fillna(0).to_numpy()
    total = probs.sum()
    if total == 0:
        return pd.Series([pd.NA] * len(group), index=group.index)
    return pd.Series(
        np.random.choice(CIVIL_LABELS, size=len(group), p=probs / total),
        index=group.index
    )


has_prov = df_users_geo["prov_code"].notna()
df_users_geo["civil_status"] = pd.NA

df_users_geo.loc[has_prov, "civil_status"] = (
    df_users_geo.loc[has_prov]
    .groupby("mun_code_base", group_keys=False)
    .apply(sample_civil_by_prov)
)

# Fallback: users without mun_code_base (CP not covered), assign by prov_code
mask_civil_fallback = df_users_geo["civil_status"].isna() & df_users_geo["prov_code"].notna()
if mask_civil_fallback.any():
    for prov in df_users_geo.loc[mask_civil_fallback, "prov_code"].dropna().unique():
        rates = df_prov_enriched.loc[df_prov_enriched["prov_code"] == prov, CIVIL_COLS]
        if rates.empty:
            continue
        probs = rates.iloc[0].fillna(0).to_numpy()
        total = probs.sum()
        if total == 0:
            continue
        idx = mask_civil_fallback & (df_users_geo["prov_code"] == prov)
        df_users_geo.loc[idx, "civil_status"] = np.random.choice(
            CIVIL_LABELS, size=idx.sum(), p=probs / total
        )

print(df_users_geo["civil_status"].value_counts())

## Car Ownership

In [ ]:
# Bernoulli(prob_car) per municipality/district (prob_car comes from municipios.csv)
has_prob_car = df_users_geo["prob_car"].notna()

df_users_geo["tiene_coche"] = pd.NA
df_users_geo.loc[has_prob_car, "tiene_coche"] = (
    np.random.random(has_prob_car.sum())
    < df_users_geo.loc[has_prob_car, "prob_car"].to_numpy()
)

print(df_users_geo["tiene_coche"].value_counts())

## Household Size

In [ ]:
# Normalise BCN labels to the numeric format ('1 persona', '2 personas', etc.)
BCN_HH_MAP = {
    "Una persona":          "1 persona",
    "Dos personas":         "2 personas",
    "Tres personas":        "3 personas",
    "Cuatro personas":      "4 personas",
    "Cinco personas":       "5 o más personas",
    "Seis personas":        "5 o más personas",
    "Siete personas":       "5 o más personas",
    "Ocho personas":        "5 o más personas",
    "Nueve o más personas": "5 o más personas",
}

df_bcn_hh = df_bcn_household.copy()
df_bcn_hh["household_size"] = df_bcn_hh["household_size"].map(BCN_HH_MAP)
df_bcn_hh = df_bcn_hh.groupby(["mun_code", "household_size"], as_index=False)["count"].sum()
df_bcn_hh["prob"] = (
    df_bcn_hh["count"] / df_bcn_hh.groupby("mun_code")["count"].transform("sum")
)

# MAD: from wide to long format
cols_hh = ["1 persona", "2 personas", "3 personas", "4 personas", "5 o más personas"]
df_mad_hh = df_mad_household.melt(
    id_vars=["mun_code"], value_vars=cols_hh,
    var_name="household_size", value_name="prob"
)

# Combine BCN + MAD + all municipalities
df_all_hh = pd.concat([
    df_bcn_hh[["mun_code", "household_size", "prob"]],
    df_mad_hh[["mun_code", "household_size", "prob"]],
    df_mun_household[["mun_code", "household_size", "prob"]],
], ignore_index=True)

# Lookup dict: mun_code -> (sizes, probabilities)
HH_LOOKUP = {}
for code, group in df_all_hh.groupby("mun_code"):
    valid = group.dropna(subset=["prob"])
    probs = valid["prob"].to_numpy(dtype=float)
    total = probs.sum()
    if total > 0:
        HH_LOOKUP[int(code)] = (valid["household_size"].tolist(), probs / total)

print(f"Municipios con distribución de hogar: {len(HH_LOOKUP)}")

In [ ]:
def sample_hogar_by_mun(group):
    code = group.name
    if pd.isna(code) or int(code) not in HH_LOOKUP:
        return pd.Series([pd.NA] * len(group), index=group.index)
    sizes, probs = HH_LOOKUP[int(code)]
    return pd.Series(
        np.random.choice(sizes, size=len(group), p=probs),
        index=group.index
    )


df_users_geo["size_hogar"] = pd.NA

# First attempt: with exact mun_code (district level for BCN/MAD)
df_users_geo["size_hogar"] = (
    df_users_geo
    .groupby("mun_code", group_keys=False)
    .apply(sample_hogar_by_mun)
)

# Fallback: with mun_code_base (city level)
missing = df_users_geo["size_hogar"].isna()
if missing.any():
    df_users_geo.loc[missing, "size_hogar"] = (
        df_users_geo.loc[missing]
        .groupby("mun_code_base", group_keys=False)
        .apply(sample_hogar_by_mun)
    )

# National fallback: aggregate distribution over all municipalities
missing = df_users_geo["size_hogar"].isna()
if missing.any():
    nacional = df_mun_household.groupby("household_size")["prob"].mean()
    nacional = nacional / nacional.sum()
    df_users_geo.loc[missing, "size_hogar"] = np.random.choice(
        nacional.index.tolist(), size=missing.sum(), p=nacional.to_numpy()
    )

print(df_users_geo["size_hogar"].value_counts())
print(f"Sin size_hogar: {df_users_geo['size_hogar'].isna().sum()}")

## Number of Rooms

In [ ]:
HAB_COLS   = ["rate_hab_menos3", "rate_hab_3a6", "rate_hab_7mas"]
HAB_LABELS = ["menos_3_hab",     "3_a_6_hab",    "7_mas_hab"]


def sample_habitaciones_by_prov(group):
    prov  = group["prov_code"].iloc[0]
    rates = df_prov_enriched.loc[df_prov_enriched["prov_code"] == prov, HAB_COLS]
    if rates.empty:
        return pd.Series([pd.NA] * len(group), index=group.index)
    probs = rates.iloc[0].fillna(0).to_numpy()
    total = probs.sum()
    if total == 0:
        return pd.Series([pd.NA] * len(group), index=group.index)
    return pd.Series(
        np.random.choice(HAB_LABELS, size=len(group), p=probs / total),
        index=group.index
    )


df_users_geo["num_room"] = pd.NA

df_users_geo.loc[has_prov, "num_room"] = (
    df_users_geo.loc[has_prov]
    .groupby("mun_code_base", group_keys=False)
    .apply(sample_habitaciones_by_prov)
)

# Fallback: users without mun_code_base (CP not covered), assign by prov_code
mask_hab_fallback = df_users_geo["num_room"].isna() & df_users_geo["prov_code"].notna()
if mask_hab_fallback.any():
    for prov in df_users_geo.loc[mask_hab_fallback, "prov_code"].dropna().unique():
        rates = df_prov_enriched.loc[df_prov_enriched["prov_code"] == prov, HAB_COLS]
        if rates.empty:
            continue
        probs = rates.iloc[0].fillna(0).to_numpy()
        total = probs.sum()
        if total == 0:
            continue
        idx = mask_hab_fallback & (df_users_geo["prov_code"] == prov)
        df_users_geo.loc[idx, "num_room"] = np.random.choice(
            HAB_LABELS, size=idx.sum(), p=probs / total
        )

print(df_users_geo["num_room"].value_counts())

---
# Validation

In [ ]:
employed = df_users_geo["labor_status"] == "employed"
inactive = df_users_geo["labor_status"] == "inactive"

print(f"Usuarios con distrito asignado:  {df_users_geo['district_name'].notna().sum()} de {len(df_users_geo)}")
print(f"Usuarios con prob_car:           {df_users_geo['prob_car'].notna().sum()}")
print(f"Usuarios con civil_status:       {df_users_geo['civil_status'].notna().sum()}")
print(f"Usuarios con size_hogar:         {df_users_geo['size_hogar'].notna().sum()}")
print(f"Usuarios con num_room:           {df_users_geo['num_room'].notna().sum()}")
print()
print("labor_status:") ; print(df_users_geo["labor_status"].value_counts())
print("\ncivil_status:") ; print(df_users_geo["civil_status"].value_counts())
print("\ntiene_coche:")  ; print(df_users_geo["tiene_coche"].value_counts())
print("\nsize_hogar:")   ; print(df_users_geo["size_hogar"].value_counts())
print("\nnum_room:")     ; print(df_users_geo["num_room"].value_counts())

---
# Save Data

## Geographic Classifications (IPA, Municipality Type, Distance)

In [ ]:
# Income class (IPA)
def classify_ipa(ipa):
    if pd.isna(ipa): return 2    # Medium income by default
    elif ipa < 85:   return 0    # Low income
    elif ipa < 100:  return 1    # Low-medium income
    elif ipa < 125:  return 2    # Medium income
    elif ipa < 150:  return 3    # Medium-high income
    else:            return 4    # High income

# Municipality type by population
def classify_mun_type(pop):
    if pd.isna(pop):   return 2  # Village (Villa) by default
    elif pop < 500:    return 0  # Hamlet (Aldea)
    elif pop < 2000:   return 1  # Small town (Pueblo)
    elif pop < 10000:  return 2  # Village (Villa)
    elif pop < 50000:  return 3  # Town (Ciudad)
    elif pop < 100000: return 4  # Large town (Gran ciudad)
    else:              return 5  # Metropolis (Metrópoli)

# Distance type to the provincial capital
def classify_distance(km):
    if pd.isna(km):  return 2   # Medium distance by default
    elif km <= 10:   return 0   # Very close
    elif km <= 25:   return 1   # Close
    elif km <= 50:   return 2   # Medium distance
    elif km <= 100:  return 3   # Far
    else:            return 4   # Very far

df_users_geo["ipa_class"]     = df_users_geo["ipa_value"].apply(classify_ipa).astype(int)
df_users_geo["mun_type"]      = df_users_geo["total_people"].apply(classify_mun_type).astype(int)
df_users_geo["distance_type"] = df_users_geo["distance_km"].apply(classify_distance).astype(int)

_ipa_lbl  = {0:"Renta baja", 1:"Renta baja-media", 2:"Renta media", 3:"Renta media-alta", 4:"Renta alta"}
_mun_lbl  = {0:"Aldea", 1:"Pueblo", 2:"Villa", 3:"Ciudad", 4:"Gran ciudad", 5:"Metrópoli"}
_dist_lbl = {0:"Muy cercana", 1:"Cercana", 2:"Media distancia", 3:"Lejana", 4:"Muy lejana"}

print("ipa_class:")
print(df_users_geo["ipa_class"].value_counts().sort_index().rename(_ipa_lbl))
print("\nmun_type:")
print(df_users_geo["mun_type"].value_counts().sort_index().rename(_mun_lbl))
print("\ndistance_type:")
print(df_users_geo["distance_type"].value_counts().sort_index().rename(_dist_lbl))

In [ ]:
df_users_enriched = df_users_geo[[
    "id_user", "email_clean", "birthday", "age", "gender", "cp_num",
    "mun_code", "prov_code",
    "mun_name", "district_name",
    "labor_status", "labor_type",
    "civil_status",
    "tiene_coche",
    "size_hogar",
    "num_room",
    "ipa_class",
    "mun_type",
    "distance_type",
]].copy()

df_users_enriched["prov_code"] = pd.to_numeric(df_users_enriched["prov_code"], errors="coerce").astype("Int64")

In [ ]:

df_users_enriched.to_csv(PROCESSED_PATH / "users.csv", index=False)

print(f"Guardado: users.csv  →  {df_users_enriched.shape}")
display(df_users_enriched.head())